In [1]:
# ============================================================
# CELL 0 — BOOTSTRAP  (identical across all pipeline notebooks)
# Finds the repo root (folder containing .env), puts src/ on the import
# path, loads shared config + download-log helpers. Machine-agnostic:
# paths come from .env via config.py, never hardcoded.
# ============================================================
import sys                                     # to modify the module search path at runtime
from pathlib import Path                        # portable path handling across Mac/PC

# Walk up from the CWD until the folder containing '.env' (the repo root) is found —
# this is what lets the same notebook run on any machine without editing paths.
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists())
sys.path.insert(0, str(_root / 'src'))          # make 'import config' / 'import download_log' resolve

from config import *                            # PROJECT_ROOT, RAW_DIR, PROCESSED_DIR, CURRENT_YEAR, ...
from download_log import load_log, update_entry, print_entry, print_stale_sources
from datetime import datetime                   # for any runtime date handling
import pandas as pd                             # primary data-handling library

log = load_log()                                # load the download-log ledger

# --- Verify the bootstrap resolved correctly before proceeding ---
print("PROJECT_ROOT :", PROJECT_ROOT)
print("RAW_DIR      :", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("CURRENT_YEAR :", CURRENT_YEAR)

PROJECT_ROOT : C:\Users\mjbou\governance-framework
RAW_DIR      : C:\Users\mjbou\governance-framework\data\raw
PROCESSED_DIR: C:\Users\mjbou\governance-framework\data\processed
CURRENT_YEAR : 2026


# Notebook 40 — ASCOR (Assessing Sovereign Climate-related Opportunities and Risks)

**Concept 12 (environmental & climate governance) — climate-policy leg. Tier: leans P2, confirm at Step 1.**

Builds a **climate-governance composite** from the TPI Centre's ASCOR sovereign assessment. ASCOR
publishes **no composite score** (its hierarchy tops out at 14 areas), so the framework constructs
its own — see `framework_decisions.md` → "ASCOR composite specification (2026-07-22)".

**Method — 9 equally-weighted areas, share of applicable indicators answered Yes:**
`EP.1` (emissions trend — the only *de facto* leg) + `CP.1`–`CP.6` (climate legislation, carbon
pricing, fossil fuels, sectoral transitions, adaptation, just transition) + `CF.2`/`CF.3`
(transparency in climate costing and spending). Excluded: `EP.2`–`EP.4` (ambition-benchmarking
against a 1.5°C fair share, not governance quality), `CF.1` (structurally a donor question),
`CF.4` (no Yes/No indicators). Exempt / Not applicable / No data are **excluded from denominators**,
not scored as No; areas with no applicable indicators drop and remaining areas renormalize.

**Scale — fixed 0–1 anchor, passed through unnormalized** (`§5` fixed-anchor family). ASCOR's
sample is advanced-economy-skewed (85 countries, 50 high-income), so distributional normalization
would penalize EMs for *being measured* alongside rich countries. The anchor is theoretical
(0 = no applicable indicator met, 1 = all met), so scores are independent of sample composition.

**Source & currency:** manual, **email-gated** download (request via the ASCOR tool page; the
export arrived automatically). Annual, three rounds to date — 2023 (25 countries), 2024 (70),
2025 (85). **Momentum is computable for only 25 of 85** countries (needs ≥3 panel points).
Vintage is derived from the data (`Assessment date`), never hardcoded. Licence **CC BY-NC 4.0**
(non-commercial — flagged for the framework-wide licence audit). **v1.2 methodology; a v2.0
overhaul is in progress** (consultation closed Jan 2026) — this build targets v1.2 and Cell 4
fails loudly if the area codes stop resolving.

In [9]:
# ============================================================
# CELL 2 — CONFIG  [REVISED 2026-07-22: two area sets, see below]
# Source id, input/output paths, and the judgment constants. Everything else —
# assessment dates, country list, area membership, indicator membership — is DERIVED
# from the files in later cells, so an annual ASCOR refresh needs no code edit: drop
# the new export in data/raw under the same filenames and re-run.
# ============================================================
import os

SOURCE_ID = "ASCOR"   # TPI Centre — Assessing Sovereign Climate-related Opportunities and Risks
                      # NOTE: confirm this id is absent from source_registry.csv (owned by nb 02).

# --- Inputs: manual, email-gated download (no fetch cell — see instructions doc) -----
# Filenames are DELIBERATELY undated so next year's export overwrites in place with no
# code change. Vintage is carried inside the data ('Assessment date') and derived in Cell 5.
RAW_RESULTS    = os.path.join(RAW_DIR, "ascor_assessments_results.xlsx")  # Yes/No indicator responses (wide)
RAW_COUNTRIES  = os.path.join(RAW_DIR, "ascor_countries.xlsx")            # ISO3 codes + income/region groupings
RAW_INDICATORS = os.path.join(RAW_DIR, "ascor_indicators.xlsx")           # area/indicator/metric dictionary
# Not used by this pipeline (emissions-pathway metrics, retained in raw for reference):
#   ascor_benchmarks.xlsx, ascor_assessments_results_trends_pathways.xlsx

OUTPUT_CSV = os.path.join(PROCESSED_DIR, "ascor_clean.csv")               # pipeline output (PANEL: country x year)

# ============================================================================
# ⚠️ MANUAL-MAINTENANCE CONSTANTS: the two area sets.
#
# WHY TWO. ASCOR deliberately asks DIFFERENT QUESTIONS OF DIFFERENT INCOME GROUPS.
# Its methodology note (Appendix 1, "Exemptions by country group") exempts its "LI"
# group — World Bank lower-middle AND low income — from the more ambitious policy
# indicators, implementing the UNFCCC common-but-differentiated-responsibilities
# principle. Separately, UNFCCC *developed* countries are exempt from transparency-in-
# climate-costing, because the Paris Agreement does not require them to disclose
# finance needs. Verified in the 2025 data: all 14 lower-middle/low-income countries
# are blanket-Exempt on carbon pricing, fossil fuels and sectoral transitions.
#
# CONSEQUENCE. Scoring each country on its own question set is NOT comparable across
# income groups, and the areas differ sharply in difficulty (fossil fuels averages
# 0.22; transparency-in-costing 0.61). Renormalizing over present areas compresses a
# real 0.108 governance gap down to 0.023 and inverts the income ordering (lower-
# middle scoring ABOVE upper-middle). See framework_decisions.md.
#
# RESOLUTION (option C). Score on the areas EVERY country answers; keep the full set
# as a diagnostic. The scored metric is comparable by construction; the diagnostic
# preserves the richer content for the evidentiary layer and for Step-4 checks.
# ============================================================================

# --- SCORED: areas with 0% blanks — every country, every round -----------------------
#   EP.1  Emissions Trends      - has the emissions profile actually improved? The ONLY
#                                 de facto leg; the other four ask whether a document exists
#   CP.1  Climate Legislation   - framework climate law + accountability elements
#   CP.5  Adaptation            - NAP, risk assessments, M&E report, early warning, cat risk pool
#   CP.6  Just Transition       - rights conventions, institutionalised approach, green jobs
#   CF.3  Transparency in Climate Spending - disclosed climate expenditure, budget tagging
ASCOR_SCORED_AREAS = ["EP.1", "CP.1", "CP.5", "CP.6", "CF.3"]

# --- DIAGNOSTIC ONLY: adds the four income/status-conditional areas ------------------
#   CP.2  Carbon Pricing         - LI group exempt (also covered by WB Carbon Pricing Dashboard)
#   CP.3  Fossil Fuels           - LI group exempt; hardest area in ASCOR (mean 0.22)
#   CP.4  Sectoral Transitions   - LI group exempt
#   CF.2  Transparency in Climate Costing - UNFCCC developed countries exempt
# NOT scored: including them would make the metric non-comparable across income groups.
ASCOR_DIAGNOSTIC_AREAS = ASCOR_SCORED_AREAS + ["CP.2", "CP.3", "CP.4", "CF.2"]

# --- Never used, in either set (recorded so the exclusions are explicit) -------------
#   CF.1            International Climate Finance — donor-only question; 63 of 85 exempt
#   CF.4            Renewable Energy Opportunities — contains no Yes/No indicators at all
#   EP.2/EP.3/EP.4  target areas — discriminating variance is AMBITION benchmarking
#                   against a 1.5C fair share, not governance quality

# --- METHODOLOGY CONSTANT: response vocabulary --------------------------------------
# Which raw responses count toward the denominator. 'Exempt' / 'Not applicable' / 'No data'
# are EXCLUDED (not scored as No): the data records "not assessed", NOT "absent". India is
# Exempt on the energy-efficiency question and HAS an energy-efficiency law — scoring it 0
# would record a false fact. Casing is normalized in Cell 5: the raw export contains BOTH
# 'No Data'/'No data' and 'Not applicable'/'Not Applicable'.
APPLICABLE_RESPONSES = {"yes", "no"}                                  # enter the denominator
EXCLUDED_RESPONSES   = {"exempt", "not applicable", "no data"}        # dropped from denominator

# --- Config check -------------------------------------------------------------------
for label, path in [("results", RAW_RESULTS), ("countries", RAW_COUNTRIES), ("indicators", RAW_INDICATORS)]:
    print(f"{label:11s} {'OK ' if os.path.exists(path) else 'MISSING'}  {path}")
print()
print(f"SCORED areas     ({len(ASCOR_SCORED_AREAS)}): {', '.join(ASCOR_SCORED_AREAS)}")
print(f"DIAGNOSTIC areas ({len(ASCOR_DIAGNOSTIC_AREAS)}): {', '.join(ASCOR_DIAGNOSTIC_AREAS)}")
print("output           :", OUTPUT_CSV)

results     OK   C:\Users\mjbou\governance-framework\data\raw\ascor_assessments_results.xlsx
countries   OK   C:\Users\mjbou\governance-framework\data\raw\ascor_countries.xlsx
indicators  OK   C:\Users\mjbou\governance-framework\data\raw\ascor_indicators.xlsx

SCORED areas     (5): EP.1, CP.1, CP.5, CP.6, CF.3
DIAGNOSTIC areas (9): EP.1, CP.1, CP.5, CP.6, CF.3, CP.2, CP.3, CP.4, CF.2
output           : C:\Users\mjbou\governance-framework\data\processed\ascor_clean.csv


In [10]:
# ============================================================
# CELL 3 — LOAD RAW WORKBOOKS
# Reads the three inputs and reports structure. No transformation yet — this cell
# exists so a malformed/changed export is visible BEFORE any scoring logic runs.
# Also reports all-null indicator columns: the export carries historical indicator
# VERSIONS as extra columns (e.g. 'CP.2.c.1'), populated only for the version in
# force. They are harmless — an all-null column contributes to neither the
# numerator nor the denominator in Cell 5 — but they are surfaced here so a future
# schema change doesn't hide behind them.
# ============================================================
import re

# --- Load: single 'Worksheet' sheet in each file (ASCOR export convention) -----------
results    = pd.read_excel(RAW_RESULTS)      # one row per country x assessment round (wide)
countries  = pd.read_excel(RAW_COUNTRIES)    # 85 rows: ISO3, region, income groupings
indicators = pd.read_excel(RAW_INDICATORS)   # dictionary: area / indicator / metric rows

print("results    :", results.shape,    "(rows, cols)")
print("countries  :", countries.shape)
print("indicators :", indicators.shape)
print()

# --- Assessment rounds present (the panel dimension; NOT hardcoded anywhere) ---------
# 'Assessment date' is the vintage marker. Cell 5 derives the year from it; Cell 7
# registers the latest as the download_log vintage.
rounds = (results.groupby("Assessment date")["Country"]
                 .nunique().rename("n_countries").reset_index())
print("assessment rounds found:")
print(rounds.to_string(index=False))
print()

# --- Indicator columns in the results file -------------------------------------------
# Column naming convention: 'indicator <CODE>' where CODE looks like 'CP.2.a'.
# pandas suffixes duplicate column names (.1/.2/.3), so the CODE is recovered by regex
# rather than by trusting the literal column name.
IND_COL_RE = re.compile(r"^indicator ((?:EP|CP|CF)\.\d+\.[a-z])")
ind_cols = [c for c in results.columns if IND_COL_RE.match(c)]
print(f"indicator columns: {len(ind_cols)}")

# --- Surface all-null indicator columns (historical versions) ------------------------
# Reported, not dropped: they are inert under the applicable-response filter in Cell 5.
empty_cols = [c for c in ind_cols if results[c].notna().sum() == 0]
print(f"all-null (superseded version) columns: {len(empty_cols)}")
for c in empty_cols:
    print("   ", c)
print()

# --- Dictionary structure -------------------------------------------------------------
# 'Type' distinguishes area / indicator / metric rows. Areas are the roll-up unit
# used by ASCOR_SCORED_AREAS; Cell 4 validates the selection against this.
print("dictionary row types:", dict(indicators["Type"].value_counts()))
print("areas in dictionary :", indicators.loc[indicators["Type"] == "area", "Code"].tolist())

results    : (180, 243) (rows, cols)
countries  : (85, 7)
indicators : (105, 6)

assessment rounds found:
Assessment date  n_countries
     18/08/2025           85
     23/08/2024           70
     31/10/2023           25

indicator columns: 48
all-null (superseded version) columns: 5
    indicator CP.2.c.1
    indicator CP.2.c.2
    indicator CP.2.c.3
    indicator CF.1.b.1
    indicator CF.1.b.2

dictionary row types: {'indicator': np.int64(48), 'metric': np.int64(43), 'area': np.int64(14)}
areas in dictionary : ['EP.1', 'EP.2', 'EP.3', 'EP.4', 'CP.1', 'CP.2', 'CP.3', 'CP.4', 'CP.5', 'CP.6', 'CF.1', 'CF.2', 'CF.3', 'CF.4']


In [12]:
# ============================================================
# CELL 4 — VALIDATE BOTH AREA SETS AGAINST THE DICTIONARY  [REVISED: two sets]
# The guard against a silent ASCOR restructure (v2.0 is in progress). For every area in
# ASCOR_DIAGNOSTIC_AREAS (which contains ASCOR_SCORED_AREAS), checks that it (a) exists
# in the shipped dictionary and (b) has at least one indicator column carrying data.
# FAILS LOUDLY listing anything that no longer resolves, rather than silently scoring a
# subset. Also builds the column->area mapping Cell 5 aggregates over.
# ============================================================
problems = []   # collect ALL failures and report together (house pattern), then fail safe

# --- (1) every area in EITHER set must exist in the dictionary ------------------------
dict_areas = set(indicators.loc[indicators["Type"] == "area", "Code"].astype(str))
for label, areas in [("SCORED", ASCOR_SCORED_AREAS), ("DIAGNOSTIC", ASCOR_DIAGNOSTIC_AREAS)]:
    missing = [a for a in areas if a not in dict_areas]
    if missing:
        problems.append(f"{label} area(s) absent from dictionary: {missing}")

# --- (2) map each indicator column to its parent area --------------------------------
# Code 'CP.2.a' -> area 'CP.2'. Built from the regex-extracted code, so pandas' duplicate
# column-name suffixes (.1/.2/.3) cannot corrupt the mapping.
col_area = {}
for c in ind_cols:
    code = IND_COL_RE.match(c).group(1)     # e.g. 'CP.2.a'
    col_area[c] = code.rsplit(".", 1)[0]    # e.g. 'CP.2'

# --- (3) every area needs >=1 indicator column WITH DATA ------------------------------
# An area whose columns are all null would silently vanish from the composite.
# Built for the DIAGNOSTIC set, which is a superset of the scored set.
area_cols, area_live = {}, {}
for a in ASCOR_DIAGNOSTIC_AREAS:
    cols = [c for c, ar in col_area.items() if ar == a]
    live = [c for c in cols if results[c].notna().sum() > 0]
    area_cols[a], area_live[a] = cols, live
    if not live:
        problems.append(f"area {a}: no indicator column carries data")

# --- (4) report the resolved structure ------------------------------------------------
print("SCORED areas (feed the metric — answered by every country):")
for a in ASCOR_SCORED_AREAS:
    print(f"   {a:6s} {len(area_live[a]):>2} live indicator col(s)")
print(f"   -> {sum(len(area_live[a]) for a in ASCOR_SCORED_AREAS)} live columns scored")

print("\nDIAGNOSTIC-ONLY areas (income/status-conditional — NOT scored):")
for a in [x for x in ASCOR_DIAGNOSTIC_AREAS if x not in ASCOR_SCORED_AREAS]:
    n_all, n_live = len(area_cols[a]), len(area_live[a])
    note = "" if n_all == n_live else f"  ({n_all - n_live} superseded-version col dropped)"
    print(f"   {a:6s} {n_live:>2} live indicator col(s){note}")

# --- (5) show what is excluded from BOTH sets (visibility, not a check) ---------------
never_used = sorted(dict_areas - set(ASCOR_DIAGNOSTIC_AREAS))
print(f"\nnever used ({len(never_used)}): {', '.join(never_used)}")

# --- (6) fail safe ---------------------------------------------------------------------
if problems:
    raise RuntimeError(
        "AREA VALIDATION FAILED — ASCOR structure has changed. Re-validate "
        "ASCOR_SCORED_AREAS / ASCOR_DIAGNOSTIC_AREAS (Cell 2) against the new "
        "methodology before scoring:\n  - " + "\n  - ".join(problems))
print("\nvalidation OK")

SCORED areas (feed the metric — answered by every country):
   EP.1    3 live indicator col(s)
   CP.1    2 live indicator col(s)
   CP.5    5 live indicator col(s)
   CP.6    4 live indicator col(s)
   CF.3    2 live indicator col(s)
   -> 16 live columns scored

DIAGNOSTIC-ONLY areas (income/status-conditional — NOT scored):
   CP.2    3 live indicator col(s)  (3 superseded-version col dropped)
   CP.3    4 live indicator col(s)
   CP.4    5 live indicator col(s)
   CF.2    2 live indicator col(s)

never used (5): CF.1, CF.4, EP.2, EP.3, EP.4

validation OK


In [13]:
# ============================================================
# CELL 5 — SCORE: area shares -> two composites (PANEL, all rounds)  [REVISED: option C]
#
# For each country x assessment round, each area is scored as:
#     area_score = (# indicators answered 'Yes') / (# APPLICABLE indicators)
#   where applicable = {'yes','no'}. Exempt / Not applicable / No data / null are excluded
#   from the denominator entirely — the data records "not assessed", NOT "absent".
#
# TWO composites, both an unweighted MEAN of the area scores present:
#   ascor_climate_governance  (SCORED)     - 5 universally-answered areas. Comparable across
#                                            all countries by construction; no renormalization
#                                            artifact because no area is ever missing.
#   ascor_full_diagnostic     (NOT SCORED) - 9 areas incl. the income/status-conditional ones.
#                                            Areas absent for a country drop and the rest
#                                            renormalize (D5 pattern). Evidentiary layer only.
#
# Equal weight per AREA (not per indicator): indicator counts per area reflect how
# decomposable ASCOR found the topic, not its importance.
# Both are on a FIXED 0-1 anchor (0 = no applicable indicator met, 1 = all met), independent
# of sample composition -> the scored metric enters under the S5 fixed-anchor family,
# passed through UNNORMALIZED. See framework_decisions.md.
# ============================================================
import numpy as np

def normalize_response(v):
    """Lower/strip a raw response so casing variants collapse.
    The export contains BOTH 'No Data'/'no data' and 'Not applicable'/'Not Applicable'."""
    return str(v).strip().lower()

def year_of(assessment_date):
    """Year derived from the data, never hardcoded. 'Assessment date' is 'DD/MM/YYYY';
    taking the last '/'-delimited part survives a day/month order change in the export."""
    tail = str(assessment_date).strip().split("/")[-1]
    if not tail.isdigit():
        raise ValueError(f"cannot derive year from Assessment date {assessment_date!r}")
    return int(tail)

def score_area(rec, area):
    """Share of an area's APPLICABLE indicators answered Yes; NaN if none applicable."""
    yes = applicable = 0
    for c in area_live[area]:                       # live columns only (Cell 4)
        v = normalize_response(rec[c])
        if v in APPLICABLE_RESPONSES:               # {'yes','no'} -> counts in denominator
            applicable += 1
            if v == "yes":
                yes += 1
        # everything else (exempt / not applicable / no data / nan) is skipped entirely
    return (yes / applicable) if applicable else np.nan, applicable

rows = []
for _, rec in results.iterrows():                   # one record = country x assessment round
    area_scores, area_appl = {}, {}
    for a in ASCOR_DIAGNOSTIC_AREAS:                # superset: covers both composites
        area_scores[a], area_appl[a] = score_area(rec, a)

    scored_vals = [area_scores[a] for a in ASCOR_SCORED_AREAS      if not np.isnan(area_scores[a])]
    diag_vals   = [area_scores[a] for a in ASCOR_DIAGNOSTIC_AREAS  if not np.isnan(area_scores[a])]

    rows.append({
        "country_name":    rec["Country"],
        "assessment_date": rec["Assessment date"],
        "year":            year_of(rec["Assessment date"]),
        # --- the SCORED metric: 5 universally-answered areas ---
        "ascor_climate_governance": (float(np.mean(scored_vals)) if scored_vals else np.nan),
        "ascor_n_areas_scored":     len(scored_vals),                                  # expect 5
        "ascor_n_indicators_appl":  sum(area_appl[a] for a in ASCOR_SCORED_AREAS),
        # --- DIAGNOSTIC (evidentiary layer only, never scored) ---
        "ascor_full_diagnostic":    (float(np.mean(diag_vals)) if diag_vals else np.nan),
        "ascor_n_areas_diagnostic": len(diag_vals),                                    # 6, 8 or 9
        # --- per-area detail, all 9, for drill-down ---
        **{f"ascor_area_{a.replace('.', '_').lower()}": area_scores[a] for a in ASCOR_DIAGNOSTIC_AREAS},
    })

scored = pd.DataFrame(rows)

# --- Report ---------------------------------------------------------------------------
print("panel rows:", len(scored), " | countries:", scored["country_name"].nunique())
print()
print("SCORED metric (ascor_climate_governance) by round:")
print(scored.groupby("year")["ascor_climate_governance"]
            .agg(["count", "mean", "std", "min", "max"]).round(3).to_string())
print()
print("areas contributing to the SCORED metric (must be 5 for every record):")
print(scored["ascor_n_areas_scored"].value_counts().sort_index().to_string())
print()
print("areas contributing to the DIAGNOSTIC composite (6/8/9 expected):")
print(scored["ascor_n_areas_diagnostic"].value_counts().sort_index().to_string())
print()
print("applicable indicators per record, scored metric (of 16 live):")
print(scored["ascor_n_indicators_appl"].describe().loc[["min", "25%", "50%", "75%", "max"]].to_string())

panel rows: 180  | countries: 85

SCORED metric (ascor_climate_governance) by round:
      count   mean    std   min    max
year                                  
2023     25  0.464  0.239  0.00  0.867
2024     70  0.483  0.234  0.05  0.900
2025     85  0.495  0.216  0.04  0.900

areas contributing to the SCORED metric (must be 5 for every record):
ascor_n_areas_scored
5    180

areas contributing to the DIAGNOSTIC composite (6/8/9 expected):
ascor_n_areas_diagnostic
6    28
8    81
9    71

applicable indicators per record, scored metric (of 16 live):
min    13.0
25%    14.0
50%    15.0
75%    15.0
max    16.0


In [14]:
# ============================================================
# CELL 6 — ATTACH ISO3 AND RUN STRUCTURAL INTEGRITY GUARDS
# Joins the ISO3 code from ASCOR's own country file (same source, so names match exactly
# by construction — but the join is guarded rather than assumed). Then runs structural
# checks only (identifiers, score domains, internal consistency) — NOT vintage-specific
# counts, so nothing here needs editing on a new ASCOR release. Collects ALL violations,
# reports together, then fails safe before Cell 7 writes anything.
# ============================================================
problems = []

# --- (1) attach ISO3 -------------------------------------------------------------------
# ascor_countries.xlsx carries 'Name' (matching results['Country']) and 'Country ISO code'.
iso_map = countries[["Name", "Country ISO code"]].rename(
    columns={"Name": "country_name", "Country ISO code": "country_code"})
scored = scored.merge(iso_map, on="country_name", how="left")

n_noiso = int(scored["country_code"].isna().sum())
if n_noiso:
    missing_names = sorted(scored.loc[scored["country_code"].isna(), "country_name"].unique())
    problems.append(f"{n_noiso} row(s) with no ISO3 — unmatched names: {missing_names}")

# --- (2) one row per country x assessment round ----------------------------------------
dups = scored.groupby(["country_code", "year"]).size()
dups = dups[dups > 1]
if len(dups):
    problems.append(f"duplicate country_code x year: {dups.index.tolist()}")

# --- (3) score domains: every area score and both composites must be in [0,1] or NaN ----
AREA_COLS = [f"ascor_area_{a.replace('.', '_').lower()}" for a in ASCOR_DIAGNOSTIC_AREAS]
for col in AREA_COLS + ["ascor_climate_governance", "ascor_full_diagnostic"]:
    bad = scored[(scored[col].notna()) & ((scored[col] < 0) | (scored[col] > 1))]
    if len(bad):
        problems.append(f"{col}: {len(bad)} value(s) outside [0,1]")

# --- (4) the SCORED metric must never be null and must always rest on all 5 areas -------
# This is the comparability guarantee. If it ever fails, an area we believe is universal
# has become conditional and the metric is no longer comparable across countries.
n_null = int(scored["ascor_climate_governance"].isna().sum())
if n_null:
    problems.append(f"{n_null} row(s) with a NULL scored metric")
bad_n = scored[scored["ascor_n_areas_scored"] != len(ASCOR_SCORED_AREAS)]
if len(bad_n):
    offenders = bad_n[["country_name", "year", "ascor_n_areas_scored"]].values.tolist()
    problems.append(f"{len(bad_n)} row(s) not scored on all {len(ASCOR_SCORED_AREAS)} areas: {offenders[:10]}")

# --- (5) diagnostic composite should rest on 6, 8 or 9 areas ---------------------------
# Informational guard: an unexpected count signals ASCOR changed its exemption structure.
unexpected = sorted(set(scored["ascor_n_areas_diagnostic"]) - {6, 8, 9})
if unexpected:
    problems.append(f"unexpected diagnostic area counts (exemption structure changed?): {unexpected}")

# --- (6) ISO3 codes should resolve against the harmonization spine ---------------------
# Reported, NOT fatal: ASCOR may cover an entity the spine flags as a territory, and the
# spine is the downstream authority on inclusion — this pipeline's job is to surface it.
import os as _os
_spine_path = _os.path.join(PROCESSED_DIR, "country_spine.csv")
if _os.path.exists(_spine_path):
    spine_iso = set(pd.read_csv(_spine_path)["iso3"].astype(str))
    off_spine = sorted(set(scored["country_code"].dropna().astype(str)) - spine_iso)
    print(f"ISO3 not found in country_spine.csv: {len(off_spine)}" + (f" -> {off_spine}" if off_spine else ""))
else:
    print("country_spine.csv not found — spine check skipped")

# --- (7) report and fail safe -----------------------------------------------------------
print()
print("rows:", len(scored), "| countries:", scored["country_code"].nunique(), "| years:", sorted(scored["year"].unique()))
print()
if problems:
    raise RuntimeError("STRUCTURAL GUARDS FAILED — nothing written:\n  - " + "\n  - ".join(problems))
print("all structural guards passed")

ISO3 not found in country_spine.csv: 0

rows: 180 | countries: 85 | years: [np.int64(2023), np.int64(2024), np.int64(2025)]

all structural guards passed


In [15]:
# ============================================================
# CELL 7 — WRITE CLEAN OUTPUT + REGISTER IN DOWNLOAD_LOG
# Writes ascor_clean.csv as a PANEL (country_code x year) — unlike the cross-section
# sources (brss/rti/pefa/fatf), ASCOR has three assessment rounds, so the full panel goes
# to the evidentiary layer and downstream scoring takes the latest slice.
# Vintage is derived from the data ('Assessment date'), never hardcoded.
# Does NOT touch source_registry.csv — that's owned by notebook 02 (cell provided after).
# ============================================================
AREA_COLS = [f"ascor_area_{a.replace('.', '_').lower()}" for a in ASCOR_DIAGNOSTIC_AREAS]

# --- vintage derived from the data, not typed ----------------------------------------
LATEST_YEAR      = int(scored["year"].max())
LATEST_ASSESSED  = scored.loc[scored["year"] == LATEST_YEAR, "assessment_date"].iloc[0]
N_LATEST         = int((scored["year"] == LATEST_YEAR).sum())

# --- assemble output ------------------------------------------------------------------
# country_code + year first (panel keys), then the scored metric and its diagnostics,
# then the non-scored full composite, then per-area detail for drill-down.
OUTPUT_COLS = (["country_code", "country_name", "year", "assessment_date",
                "ascor_climate_governance", "ascor_n_areas_scored", "ascor_n_indicators_appl",
                "ascor_full_diagnostic", "ascor_n_areas_diagnostic"] + AREA_COLS)
out = scored[OUTPUT_COLS].copy()
out = out[out["country_code"].notna() & (out["country_code"].astype(str).str.strip() != "")]
out = out.sort_values(["country_code", "year"]).reset_index(drop=True)
out.to_csv(OUTPUT_CSV, index=False)

print("Wrote  :", OUTPUT_CSV)
print("Shape  :", out.shape, "(rows, cols)")
print("Columns:", list(out.columns))
print(f"\nLatest round: {LATEST_ASSESSED} ({LATEST_YEAR}), {N_LATEST} countries")
print("\nSample (latest round, top of ranking on the SCORED metric):")
print(out[out["year"] == LATEST_YEAR]
        .sort_values("ascor_climate_governance", ascending=False)
        [["country_code", "country_name", "ascor_climate_governance", "ascor_full_diagnostic"]]
        .head(5).to_string(index=False))

# --- register in download_log; vintage from data, not typed ---------------------------
today_str = datetime.today().strftime("%Y-%m-%d")
update_entry(
    SOURCE_ID,                                       # ASCOR
    last_successful_download_date=today_str,
    data_as_of_date=f"{LATEST_YEAR} (assessment dated {LATEST_ASSESSED}; {N_LATEST} countries)",
    local_filename=os.path.basename(OUTPUT_CSV),     # ascor_clean.csv
    latest_available_version=f"ASCOR v1.2 methodology, {LATEST_YEAR} assessment round",
    notes=(
        "TPI Centre (LSE) ASCOR — sovereign climate assessment, Concept 12 climate-policy leg. "
        "Tier leans P2 (85 countries = 44% of the 192-sovereign core; only 35 non-high-income; "
        "Sub-Saharan Africa 6, South Asia 4) — CONFIRM at Step-1. "
        "SCORED METRIC = ascor_climate_governance: equally-weighted mean of the FIVE areas every "
        "country answers (EP.1 emissions trends, CP.1 climate legislation, CP.5 adaptation, "
        "CP.6 just transition, CF.3 transparency in climate spending), each = share of applicable "
        "indicators answered Yes. FIXED 0-1 anchor -> S5 fixed-anchor family, passed through "
        "UNNORMALIZED (ASCOR's sample is advanced-skewed, so distributional normalization would "
        "penalize EMs for being measured alongside rich countries). "
        "WHY ONLY FIVE AREAS: ASCOR deliberately asks different questions of different income "
        "groups (methodology Appendix 1) — its 'LI' group (WB lower-middle + low income) is exempt "
        "from carbon pricing / fossil fuels / sectoral transitions, and UNFCCC developed countries "
        "are exempt from transparency-in-climate-costing. Scoring each country on its own question "
        "set compresses a real 0.108 governance gap to 0.023 and inverts the income ordering. The "
        "five universal areas are comparable by construction. ascor_full_diagnostic (9 areas, "
        "renormalized) is retained for the evidentiary layer and is NOT scored. "
        "Exempt / Not applicable / No data are EXCLUDED from denominators, never scored as No — the "
        "data records 'not assessed', not 'absent'. "
        "PANEL: 3 rounds (2023/25 countries, 2024/70, 2025/85) — momentum needs >=3 points so it is "
        "computable for only 25 of 85 countries. "
        "CAUTION: ASCOR is ALREADY WEALTH-ADJUSTED by design (income-group exemptions implementing "
        "common-but-differentiated-responsibilities) — flag for the planned wealth-adjustment layer "
        "to avoid double-crediting poorer countries. "
        "MANUAL email-gated download; annual. v2.0 overhaul in progress (consultation closed Jan "
        "2026) — Cell 4 fails loudly if area codes stop resolving. CC BY-NC 4.0 (NON-COMMERCIAL — "
        "flagged for the framework-wide licence audit). See instructions_data_maintenance.md."
    ),
)
print("\n--- download_log entry ---")
print_entry(SOURCE_ID)

Wrote  : C:\Users\mjbou\governance-framework\data\processed\ascor_clean.csv
Shape  : (180, 18) (rows, cols)
Columns: ['country_code', 'country_name', 'year', 'assessment_date', 'ascor_climate_governance', 'ascor_n_areas_scored', 'ascor_n_indicators_appl', 'ascor_full_diagnostic', 'ascor_n_areas_diagnostic', 'ascor_area_ep_1', 'ascor_area_cp_1', 'ascor_area_cp_5', 'ascor_area_cp_6', 'ascor_area_cf_3', 'ascor_area_cp_2', 'ascor_area_cp_3', 'ascor_area_cp_4', 'ascor_area_cf_2']

Latest round: 18/08/2025 (2025), 85 countries

Sample (latest round, top of ranking on the SCORED metric):
country_code country_name  ascor_climate_governance  ascor_full_diagnostic
         PRT     Portugal                  0.900000               0.862500
         CHL        Chile                  0.866667               0.718519
         FIN      Finland                  0.850000               0.843750
         IRL      Ireland                  0.850000               0.814583
         FRA       France            